# CodeGen agent tests

Tests the parser → formulator → codegen chain, runs the generated script,
then checks codegen's error-handling branches without calling the LLM.
Run cells top to bottom.

**Setup:** ensure `.env` has `ANTHROPIC_API_KEY` (copy from `.env.example`).

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "orharness").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Copy .env.example to .env and add your key."
    )

print("API key loaded from .env")

API key loaded from .env


In [2]:
from orharness.models import ORHarnessConfig
from orharness.agents.parser import parse_problem
from orharness.agents.formulator import formulate_problem
from orharness.agents.codegen import generate_code, RESULT_MARKER
from orharness.sandbox import run_code

config = ORHarnessConfig()
print(config)

model='claude-sonnet-4-6' max_retries=3 timeout_seconds=30 temperature=0.0


## Test: scheduling problem (parser → formulator → codegen)

Generate an OR-Tools script for the nurse problem. Expect a CP-SAT solver,
`objective_kind=feasibility`, and `objective_value: null` when the script runs.

In [3]:
parsed = parse_problem(
    "I have 6 nurses, 3 shifts per day, 7 days a week. "
    "No nurse works more than 5 shifts per week. "
    "Night shifts need at least 2 nurses.",
    config,
)
formulated = formulate_problem(parsed, config)
generated = generate_code(formulated, config)

print("parsed objective_kind:", parsed.objective_kind)
print("formulated objective_kind:", formulated.objective_kind)
print("formulated objective:", formulated.objective)
print("solver:", generated.solver)
print("problem_type:", generated.problem_type)
print("attempt:", generated.attempt)
print("code length:", len(generated.code), "chars")
print("\n--- generated code ---\n")
print(generated.code)

parsed objective_kind: ObjectiveKind.FEASIBILITY
formulated objective_kind: ObjectiveKind.FEASIBILITY
formulated objective: None
solver: ORSolver.CP_SAT
problem_type: ProblemType.SCHEDULING
attempt: 1
code length: 1956 chars

--- generated code ---

from ortools.sat.python import cp_model
import json

def main():
    model = cp_model.CpModel()

    num_nurses = 6
    shifts_per_day = 3
    days_per_week = 7

    # x[i][j][k]: binary variable
    x = {}
    for i in range(num_nurses):
        for j in range(shifts_per_day):
            for k in range(days_per_week):
                x[i, j, k] = model.NewBoolVar(f'x[{i}][{j}][{k}]')

    # Each nurse works at most 5 shifts per week
    for i in range(num_nurses):
        model.Add(sum(x[i, j, k] for j in range(shifts_per_day) for k in range(days_per_week)) <= 5)

    # Each night shift must have at least 2 nurses
    for k in range(days_per_week):
        model.Add(sum(x[i, 2, k] for i in range(num_nurses)) >= 2)

    # Each nurse works 

## Run the generated script (sandbox)

Execute via `run_code`, then split on the result marker and parse the JSON
the way the pipeline will.

In [4]:
import json

execution = run_code(generated, config)

print("success:", execution.success)
print("sandbox wall time:", round(execution.solve_time_seconds, 3), "s")
if execution.error_message:
    print("error:", execution.error_message[:500])

stdout = execution.raw_output or ""
if RESULT_MARKER in stdout:
    payload = stdout.split(RESULT_MARKER, 1)[1].strip()
    result = json.loads(payload)
    print("\nparsed result:")
    print("  status:", result["status"])
    print("  feasible:", result["feasible"])
    print("  objective_value:", result["objective_value"])
    print("  solve_time_seconds:", result["solve_time_seconds"])
    print("  solution keys:", list(result["solution"].keys()))
else:
    print("\nNo result marker found in stdout")

exit code: 0

parsed result:
  status: OPTIMAL
  feasible: True
  objective_value: None
  solve_time_seconds: 0.007154000000000001
  solution keys: ['assignments']


## Test: codegen error handling (no LLM)

Force each failure branch in `generate_code` by swapping the LLM call
(`completion`) for a fake that returns canned bad code. Plus one branch
(unmapped problem type) that fails before any LLM call. Each must raise
`CodeGenerationError`.

In [5]:
from types import SimpleNamespace

import orharness.agents.codegen as cgmod
from orharness.models import FormulatedModel, ProblemType, ObjectiveKind
from orharness.exceptions import CodeGenerationError

dummy_formulated = FormulatedModel(
    problem_type=ProblemType.SCHEDULING,
    objective_kind=ObjectiveKind.FEASIBILITY,
    variables=["x[i]: binary"],
    objective=None,
    constraints=["c1"],
    parameters={},
)


def fake_completion_returning(content: str):
    def _fake(*args, **kwargs):
        message = SimpleNamespace(content=content)
        return SimpleNamespace(choices=[SimpleNamespace(message=message)])
    return _fake


def expect_codegen_error(label: str, fake_content: str):
    cgmod.completion = fake_completion_returning(fake_content)
    try:
        cgmod.generate_code(dummy_formulated, config)
    except CodeGenerationError as e:
        print(f"PASS [{label}] raised CodeGenerationError: {str(e)[:60]}...")
    else:
        print(f"FAIL [{label}] expected CodeGenerationError, none raised")


original_completion = cgmod.completion
try:
    expect_codegen_error("empty code", "")
    expect_codegen_error("syntax error", "from ortools.sat.python import cp_model\ndef (")
    expect_codegen_error("missing ortools", "x = 1\nprint(x)")
    expect_codegen_error(
        "feasibility with objective",
        "from ortools.sat.python import cp_model\n"
        "model = cp_model.CpModel()\n"
        "model.Minimize(0)\n",
    )

    # Unmapped problem type fails before the LLM is ever called.
    unknown_formulated = dummy_formulated.model_copy(
        update={"problem_type": ProblemType.UNKNOWN}
    )
    try:
        cgmod.generate_code(unknown_formulated, config)
    except CodeGenerationError as e:
        print(f"PASS [unmapped type] raised CodeGenerationError: {str(e)[:60]}...")
    else:
        print("FAIL [unmapped type] expected CodeGenerationError, none raised")

    # Positive control: valid code should return a GeneratedCode.
    cgmod.completion = fake_completion_returning(
        "from ortools.sat.python import cp_model\nprint('hello')"
    )
    ok = cgmod.generate_code(dummy_formulated, config)
    print(f"PASS [valid code] -> solver={ok.solver}, {len(ok.code)} chars")
finally:
    cgmod.completion = original_completion  # restore the real LLM call

PASS [empty code] raised CodeGenerationError: Code generator returned empty code...
PASS [syntax error] raised CodeGenerationError: Code generator returned code with a syntax error: invalid sy...
PASS [missing ortools] raised CodeGenerationError: Generated code does not import ortools...
PASS [feasibility with objective] raised CodeGenerationError: Feasibility problem must not set an objective in generated c...
PASS [unmapped type] raised CodeGenerationError: No solver mapping for problem type: ProblemType.UNKNOWN...
PASS [valid code] -> solver=ORSolver.CP_SAT, 54 chars
